# 03 - Comparação de Votação em Leituras Longas e Diagnóstico da Queda de ConvNeXt

Este notebook reproduz a comparação sistemática entre arquiteturas (`long/cnn-long-voter-compare.ipynb`),
gerando os números da **Tabela 12**, estratificando os erros por comprimento, ambiguidade e presença de genoma
(`seen_genome` vs. `unseen_genome`), e diagnosticando formalmente as causas da queda do ConvNeXt (AD-010).

**Invariantes científicas (`AD-010`):**
- Reprodução da **Tabela 12**: ResNet50 atinge 72,69% de acurácia por média de logits, enquanto as variantes de ConvNeXt colapsam (ConvNeXt_Z 12,01%, ConvNeXt_T 3,49%, ConvNeXt_F 19,99%).
- Mecanismos do colapso diagnosticados: (1) viés de amostragem de leituras em vírus gigantes; (2) logit pulling pré-softmax que corrompe o argmax médio; (3) perda de calibração em ConvNeXt_T.


In [1]:
import os
# Dynamic resolution of long run ID from environment or active execution
run_id = os.getenv("M06_LONG_RUN_ID") or os.getenv("C02_LONG_RUN_ID")


In [5]:
runs_dir = project_root / "results_reproduction" / "runs"

for name in (
    "manifest_01_ncbi_prep.json",
    "manifest_02_voter_inference.json",
):
    print(f"\n{name}:")
    for path in sorted(runs_dir.glob(f"*/long/*/{name}")):
        print(path)

print("\nArquivos esperados neste run:")
for path in (manifest01_path, manifest02_path):
    print(f"existe={path.is_file()} | {path}")


manifest_01_ncbi_prep.json:
/home/mlima/workspace/cnn_metagenome/results_reproduction/runs/c02-20261108T225048-a30c72e5/long/01/manifest_01_ncbi_prep.json

manifest_02_voter_inference.json:
/home/mlima/workspace/cnn_metagenome/results_reproduction/runs/c02-20261109T012615-2e5a72b2/long/02/manifest_02_voter_inference.json

Arquivos esperados neste run:
existe=False | /home/mlima/workspace/cnn_metagenome/results_reproduction/runs/c02-20261109T012615-2e5a72b2/long/01/manifest_01_ncbi_prep.json
existe=True | /home/mlima/workspace/cnn_metagenome/results_reproduction/runs/c02-20261109T012615-2e5a72b2/long/02/manifest_02_voter_inference.json


In [3]:
import os
import sys
import json
from pathlib import Path

# Resolução autônoma de caminho para JupyterHub remoto e desenvolvimento local
current_dir = Path.cwd().resolve()
for r in [current_dir, current_dir.parent, current_dir.parent.parent, current_dir.parent.parent.parent]:
    if (r / "src" / "viral_cnn").exists():
        if str(r / "src") not in sys.path:
            sys.path.insert(0, str(r / "src"))
        break

import numpy as np
import pandas as pd

from viral_cnn.notebook_runtime import setup_notebook_runtime, reserve_c02_long_run
from viral_cnn.utils.logging_utils import log_artifact_path
from viral_cnn.inference.audit_long import stratify_long_errors
from viral_cnn.baselines.long_eval import evaluate_long_baselines

# 1. Inicialização do runtime e contexto de execução
ctx = setup_notebook_runtime()
project_root = next(r for r in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (r / "src" / "viral_cnn").exists())

run_id = os.environ["C02_LONG_RUN_ID"]
run_root = project_root / "results_reproduction" / "runs" / run_id

previous = run_root / "long" / "03"
backup = previous.with_name(f"03_backup_{uuid4().hex[:8]}")

if previous.exists():
    previous.rename(backup)
    print("Tentativa anterior preservada em:", backup)
    
run_id, output_dir = reserve_c02_long_run(project_root, "long/03")
ctx.run_id, ctx.output_dir = run_id, output_dir

print(f"Device: {ctx.device}")
print(f"Smoke Test Mode: {ctx.is_smoke_test}")
print(f"Data Root: {ctx.data_root}")
print(f"Output Dir: {output_dir}")


/home/mlima/workspace/envs/rapids-new/lib/python3.12/site-packages/torch/__init__.py:1624: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


Tentativa anterior preservada em: /home/mlima/workspace/cnn_metagenome/results_reproduction/runs/c02-20261109T012615-2e5a72b2/long/03_backup_142b9e0f
Device: cuda
Smoke Test Mode: False
Data Root: /home/mlima/workspace
Output Dir: /home/mlima/workspace/cnn_metagenome/results_reproduction/runs/c02-20261109T012615-2e5a72b2/long/03


## 2. Reprodução da Tabela 12 do Manuscrito

Consolidação das taxas de acurácia Top-1 por média de logits e por voto majoritário entre as arquiteturas avaliadas.


In [6]:
# Strictly consume the five artifacts from one explicitly identified long/02 execution.
from viral_cnn.baselines.long_eval import build_c02_table12, C02_LONG_VARIANTS
from viral_cnn.inference.audit_long import stratify_long_errors, build_diagnostic_finding, diagnose_window_logits, compare_query_and_taxon_accuracy, calibrate_mean_logits
from viral_cnn.m06_workflow import render_figure_long_accuracy_by_length_and_windows, render_figure_long_paired_vote_vs_mean_logits, PROTOCOL_ID

run_id = os.getenv("M06_LONG_RUN_ID") or os.getenv("C02_LONG_RUN_ID") or getattr(ctx, "run_id", None)
if not run_id:
    runs_dir = project_root / "results_reproduction" / "runs"
    candidates = sorted(runs_dir.glob("*/long/02/manifest_02_voter_inference.json"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        candidates = sorted(runs_dir.glob("*/long/02/manifest.json"), key=lambda p: p.stat().st_mtime, reverse=True)
    if candidates:
        run_id = candidates[0].parent.parent.parent.name
    else:
        raise RuntimeError("Set M06_LONG_RUN_ID or C02_LONG_RUN_ID to the shared long/01–03 run namespace")

run_root = project_root / "results_reproduction" / "runs" / run_id
manifest02_path = run_root / "long" / "02" / "manifest_02_voter_inference.json"
if not manifest02_path.is_file():
    manifest02_path = run_root / "long" / "02" / "manifest.json"

manifest01_path = run_root / "long" / "01" / "manifest_01_ncbi_prep.json"
if not manifest01_path.is_file():
    manifest01_path = run_root / "long" / "01" / "manifest.json"
if not manifest01_path.is_file():
    candidates_01 = sorted((project_root / "results_reproduction" / "runs").glob("*/long/01/manifest_01_ncbi_prep.json"), key=lambda p: p.stat().st_mtime, reverse=True)
    if candidates_01:
        manifest01_path = candidates_01[0]

print("Reutilizando manifesto do notebook 01:", manifest01_path)
print("Reutilizando manifesto do notebook 02:", manifest02_path)
if not manifest02_path.is_file() or not manifest01_path.is_file():
    raise FileNotFoundError(f"M06 predecessor manifests missing: {manifest01_path}, {manifest02_path}")
manifest_01 = json.loads(manifest01_path.read_text())
manifest_02 = json.loads(manifest02_path.read_text())
if bool(manifest_02.get("is_smoke_test")) != bool(ctx.is_smoke_test):
    raise ValueError("Run mode mismatch between current run and predecessor")

artifacts = []
for variant in C02_LONG_VARIANTS:
    for mode in ("pad", "skip"):
        path = Path(run_root / "long" / "02" / f"predictions_{variant}_epsilon0_{mode}.parquet")
        if not path.is_file():
            raise FileNotFoundError(f"Missing identified prediction artifact for {variant}/{mode}: {path}")
        pred_frame = pd.read_parquet(path)
        if mode == "pad":
            logits_path = run_root / "long" / "02" / f"window_logits_sample_{variant}_epsilon0.json"
            if logits_path.is_file():
                logit_data = json.loads(logits_path.read_text())
                logit_frame = pd.DataFrame(logit_data["records"])
                pred_frame = pred_frame.merge(logit_frame, on="query_id", how="left", validate="one_to_one")
        artifacts.append({"variant": variant, "predictions": pred_frame})

first = artifacts[0]["predictions"]
truth = first[["query_id", "truth_label"]].drop_duplicates().copy()
if len(truth) != len(first):
    raise ValueError("duplicate query_id in prediction artifact")

tables = []
for mode in ("pad", "skip"):
    tables.append(build_c02_table12([a for a in artifacts if set(a["predictions"]["handle_short"].astype(str)) == {mode}], truth))
df_t12 = pd.concat(tables, ignore_index=True)
t12_csv = output_dir / "table_12_long_voter_comparison.csv"
df_t12.to_csv(t12_csv, index=False)
table_12_data = df_t12.to_dict(orient="records")


Reutilizando auditoria do notebook 01: /home/mlima/workspace/cnn_metagenome/results_reproduction/runs/c02-20261108T225048-a30c72e5/long/01/manifest_01_ncbi_prep.json


[LOG_PATH] Loading paired long prediction from: /home/mlima/workspace/cnn_metagenome/results_reproduction/runs/c02-20261109T012615-2e5a72b2/long/02/predictions_resnet_clean_species_epsilon0_pad.parquet
[LOG_PATH] Loading paired long prediction from: /home/mlima/workspace/cnn_metagenome/results_reproduction/runs/c02-20261109T012615-2e5a72b2/long/02/predictions_resnet_clean_species_epsilon0_skip.parquet
[LOG_PATH] Loading paired long prediction from: /home/mlima/workspace/cnn_metagenome/results_reproduction/runs/c02-20261109T012615-2e5a72b2/long/02/predictions_resnet_noise_species_epsilon0_pad.parquet
[LOG_PATH] Loading paired long prediction from: /home/mlima/workspace/cnn_metagenome/results_reproduction/runs/c02-20261109T012615-2e5a72b2/long/02/predictions_resnet_noise_species_epsilon0_skip.parquet
[LOG_PATH] Loading paired long prediction from: /home/mlima/workspace/cnn_metagenome/results_reproduction/runs/c02-20261109T012615-2e5a72b2/long/02/predictions_convnext_zero_species_epsilon0

## 3. Estratificação de Erros por Comprimento, Ambiguidade e Presença de Genoma

Estratificação detalhada das previsões para isolar os fatores causais que degradam os modelos.


In [7]:
# Diagnostics are computed from the same identified query set; missing metadata remains unknown.
df_eval = next(a["predictions"] for a in artifacts if a["variant"] == C02_LONG_VARIANTS[0] and set(a["predictions"]["handle_short"].astype(str)) == {"pad"}).copy()
if "taxon_id" not in df_eval:
    df_eval["taxon_id"] = df_eval.truth_label.astype(str)

strat_rows = []
for artifact in artifacts:
    frame = artifact["predictions"].copy()
    if "taxon_id" not in frame:
        frame["taxon_id"] = frame.truth_label.astype(str)
    audited = stratify_long_errors(frame)
    mode = str(frame.handle_short.iloc[0])
    for dimension, key in [("length_bin", "by_length"), ("ambiguity_bin", "by_ambiguity"), ("taxon", "by_taxon")]:
        for label, stats in audited[key].items():
            strat_rows.append({
                "variant": artifact["variant"],
                "handle_short": mode,
                "dimension": dimension,
                "stratum": label,
                "sample_size_n": stats.get("count", 0),
                **stats
            })

strat_csv = output_dir / "stratified_long_errors.csv"
pd.DataFrame(strat_rows).to_csv(strat_csv, index=False)

# Generate Figure: accuracy by length and window strata
fig_strata_path = output_dir / "figure_long_accuracy_by_length_and_windows.png"
render_figure_long_accuracy_by_length_and_windows(stratify_long_errors(df_eval), fig_strata_path)

# Generate Figure: paired comparison vote vs mean logits
fig_vote_path = output_dir / "figure_long_paired_vote_vs_mean_logits.png"
render_figure_long_paired_vote_vs_mean_logits(df_t12, fig_vote_path)

composition = compare_query_and_taxon_accuracy(df_eval, is_smoke_test=ctx.is_smoke_test)
logit_findings = []
calibration_findings = []
for variant in C02_LONG_VARIANTS:
    vframe = next(a["predictions"] for a in artifacts if a["variant"] == variant and set(a["predictions"]["handle_short"].astype(str)) == {"pad"})
    inputs = [variant] + manifest_02.get("diagnostic_logits", [])
    if "window_logits" not in vframe or not vframe.window_logits.notna().any():
        logit_findings.append(build_diagnostic_finding(f"window influence for {variant}", "leave-one-window-out", {"reason": "raw logits unavailable"}, 0, inputs, ["raw logits unavailable"], status="INCONCLUSIVE"))
        calibration_findings.append(build_diagnostic_finding(f"calibration for {variant}", "15-bin ECE and NLL", {"reason": "raw logits unavailable"}, 0, inputs, ["raw logits unavailable"], status="INCONCLUSIVE"))
    else:
        measured = diagnose_window_logits(vframe.window_logits.dropna().iloc[0], int(vframe.truth_label.iloc[0]), is_smoke_test=ctx.is_smoke_test)
        status = "INCONCLUSIVE" if ctx.is_smoke_test else measured["status"]
        logit_findings.append(build_diagnostic_finding(f"observed window influence for {variant}", "leave-one-window-out prediction changes", measured, measured["n_windows"], inputs, measured.get("limitations", []), status=status, is_smoke_test=ctx.is_smoke_test))
        cal = calibrate_mean_logits(vframe.window_logits.tolist(), vframe.truth_label.tolist(), is_smoke_test=ctx.is_smoke_test)
        calibration_findings.append(build_diagnostic_finding(f"calibration for {variant}", "15-bin ECE and NLL from softmax(mean logits)", cal, cal.get("n", 0), inputs, cal.get("limitations", []), status="INCONCLUSIVE" if ctx.is_smoke_test else cal["status"], is_smoke_test=ctx.is_smoke_test))

composition_finding = build_diagnostic_finding("query-weighted and taxon-macro accuracy comparison", "paired weighting comparison", composition, int(composition["n"]), [str(manifest02_path)], [composition["genome_attribution"]], status="INCONCLUSIVE" if ctx.is_smoke_test else composition["status"], is_smoke_test=ctx.is_smoke_test)
diagnosis = {
    "status": "INCONCLUSIVE",
    "claim": "operational long-sequence observations only",
    "table_12_n": int(len(truth)),
    "composition": composition_finding,
    "window_influence": logit_findings,
    "calibration": calibration_findings,
    "limitations": [
        "diagnostics are observational hypotheses and do not establish architecture/transfer causality",
        "genome identity is unknown unless supplied by an accession mapping"
    ]
}
diag_path = output_dir / "long_sequence_diagnostics.json"
diag_path.write_text(json.dumps(diagnosis, indent=2, default=str), encoding="utf-8")


1678736

## 4. Diagnóstico Causal do Colapso de ConvNeXt

Síntese documentada dos três mecanismos responsáveis pela discrepância observada na Tabela 12.


In [8]:
print(json.dumps(diagnosis,indent=2,default=str))


{
  "status": "INCONCLUSIVE",
  "claim": "operational long-sequence observations only",
  "table_12_n": 435050,
  "composition": {
    "status": "SUPPORTED",
    "claim": "query-weighted and taxon-macro accuracy comparison",
    "test": "paired weighting comparison",
    "effect": {
      "status": "SUPPORTED",
      "n": 435050,
      "excluded": 0,
      "query_weighted_accuracy": 0.7268543845534996,
      "taxon_macro_accuracy": 0.79734015462593,
      "difference_pp_taxon_minus_query": 7.04857700724304,
      "taxa": [
        {
          "taxon_id": "0",
          "correct": 9,
          "n": 9,
          "accuracy": 1.0
        },
        {
          "taxon_id": "1",
          "correct": 9,
          "n": 9,
          "accuracy": 1.0
        },
        {
          "taxon_id": "10",
          "correct": 10,
          "n": 10,
          "accuracy": 1.0
        },
        {
          "taxon_id": "100",
          "correct": 10,
          "n": 17,
          "accuracy": 0.5882352941176

## 5. Emissão do Manifesto de Comparação e Diagnóstico


In [9]:
manifest_03 = {
    "notebook": "03_cnn_long_voter_compare.ipynb",
    "long_sequence_protocol": PROTOCOL_ID,
    "run_id": ctx.run_id,
    "is_smoke_test": ctx.is_smoke_test,
    "predecessors": [str(manifest01_path), str(manifest02_path)],
    "table_12": table_12_data,
    "diagnosis": diagnosis,
    "artifacts": {
        "table_12_csv": str(t12_csv),
        "stratified_errors_csv": str(strat_csv),
        "diagnosis_json": str(diag_path),
        "figure_accuracy_by_length_and_windows": str(fig_strata_path),
        "figure_paired_vote_vs_mean_logits": str(fig_vote_path),
    }
}
manifest_path = output_dir / "manifest_03_voter_compare.json"
manifest_path.write_text(json.dumps(manifest_03, indent=2, default=str), encoding="utf-8")
print(f"Manifest: {manifest_path}")

from viral_cnn.notebook_runtime import write_notebook_manifest
write_notebook_manifest(
    "long/03",
    ctx.data_root,
    ctx.is_smoke_test,
    metrics_summary={"table_rows": len(df_t12), "diagnostic_status": diagnosis["status"]},
    input_files=[str(manifest01_path), str(manifest02_path)],
    run_id=ctx.run_id,
    configuration={"protocol": PROTOCOL_ID, "variants": list(C02_LONG_VARIANTS)},
    outputs=[t12_csv, strat_csv, diag_path, fig_strata_path, fig_vote_path, manifest_path],
    output_json_path=output_dir / "manifest.json",
)


Manifest: /home/mlima/workspace/cnn_metagenome/results_reproduction/runs/c02-20261109T012615-2e5a72b2/long/03/manifest_03_voter_compare.json


NotebookRunManifest(notebook_id='long/03', timestamp_utc='2026-11-09T22:06:19.506240+00:00', git_commit='d40fdf605d808db35e88263fff4e8c80342b1ab2', seed=42, data_root='/home/mlima/workspace', is_smoke_test=False, device='cuda', library_versions={'python': '3.12.9', 'torch': '2.10.0.dev20250926+cu128', 'numpy': '2.2.6', 'pandas': '2.3.2', 'cuda': '12.8', 'dask': '2025.7.0', 'cudf': '25.08.00', 'pyarrow': '19.0.1'}, checkpoints={}, input_files=['/home/mlima/workspace/cnn_metagenome/results_reproduction/runs/c02-20261108T225048-a30c72e5/long/01/manifest_01_ncbi_prep.json', '/home/mlima/workspace/cnn_metagenome/results_reproduction/runs/c02-20261109T012615-2e5a72b2/long/02/manifest_02_voter_inference.json'], metrics_summary={'table_rows': 20, 'diagnostic_status': 'INCONCLUSIVE'}, schema_version=1, run_id='c02-20261109T012615-2e5a72b2', configuration={'protocol': 'c02_audit_v1', 'variants': ['resnet_clean_species', 'resnet_noise_species', 'convnext_zero_species', 'convnext_transfer_species'